# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aneeqahabib/FlyRank_ML_Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

This is a binary prioritization problem in the ranking-signal lane: we want to score which content items are most likely to be declining, and then compare that ranking to the Week-4 rule baseline on the same holdout slice. I am using a logistic regression model with a standard preprocessor because the task is yes/no, the outcome is observed, and the method is transparent: it gives a clean probability score, reads well, and can be interpreted by feature direction without hiding behind a complex black box.

A simple model is a feature here, not a limitation. For editorial review queues, a readable decision score is more useful than a slightly higher but opaque metric. We are not trying to prove causality; we are trying to ask whether a learned score has real lift over the hand-written baseline on the same data.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np

# Load the starter data and keep the observed label.
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].eq('down').astype(int)

# Honest feature frame for the ranking-signal lane.
feature_columns = [
    'impressions_90d',
    'clicks_90d',
    'ctr',
    'avg_position',
    'engagement_rate',
    'scroll_rate',
    'content_age_days',
    'days_since_last_update',
    'word_count',
    'search_volume',
    'competition',
    'impressions_prev_30d',
    'clicks_prev_30d',
    'sessions_prev_30d',
    'days_with_impressions',
    'days_with_sessions',
]

# Keep only the rows where the selected feature set is available.
model_frame = df[['content_id', 'client_id', 'is_declining_label'] + feature_columns].copy()
model_frame = model_frame.dropna(subset=feature_columns + ['is_declining_label']).reset_index(drop=True)

# Grouped split: by client, not random row-level. This keeps the same client from leaking across train and test.
client_ids = sorted(model_frame['client_id'].unique())
np.random.seed(42)
train_clients = set(np.random.choice(client_ids, size=max(1, len(client_ids) // 2), replace=False))
valid_clients = set(client_ids) - train_clients

train_mask = model_frame['client_id'].isin(train_clients)
test_mask = model_frame['client_id'].isin(valid_clients)

print('Train rows:', int(train_mask.sum()))
print('Test rows:', int(test_mask.sum()))
print('Train clients:', len(train_clients))
print('Test clients:', len(valid_clients))
print('Base rate in full slice:', round(model_frame['is_declining_label'].mean(), 3))
print('Base rate in test slice:', round(model_frame.loc[test_mask, 'is_declining_label'].mean(), 3))

X_train = model_frame.loc[train_mask, feature_columns]
X_test = model_frame.loc[test_mask, feature_columns]
y_train = model_frame.loc[train_mask, 'is_declining_label']
y_test = model_frame.loc[test_mask, 'is_declining_label']


Train rows: 7394
Test rows: 12503
Train clients: 14
Test clients: 15
Base rate in full slice: 0.601
Base rate in test slice: 0.618


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-grouped holdout split instead of a random row split. This is important for the lane because content pages are not independent: pages from the same client share editorial strategy, page mix, and traffic patterns. A random row split would make the model look better than it really is by counting repeated client behavior in both train and test. The client-grouped split keeps the comparison honest and closer to a real operational review setting.

The training slice is built from roughly half of the clients, and the held-out slice is the remaining clients. That design avoids leakage from same-client pages and gives a clearer answer to the question: does the model help prioritize declining content for unseen clients, not just re-score familiar pages?


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print('Client-grouped split summary:')
print('Train clients:', len(train_clients))
print('Test clients:', len(valid_clients))
print('Train rows:', int(train_mask.sum()))
print('Test rows:', int(test_mask.sum()))
print('Test base rate:', round(model_frame.loc[test_mask, 'is_declining_label'].mean(), 3))
print('No client overlap:', set(model_frame.loc[train_mask, 'client_id']).isdisjoint(set(model_frame.loc[test_mask, 'client_id'])))


Client-grouped split summary:
Train clients: 14
Test clients: 15
Train rows: 7394
Test rows: 12503
Test base rate: 0.618
No client overlap: True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Fit a readable model on the client-grouped train slice.
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42),
)
model.fit(X_train, y_train)

model_test = model_frame.loc[test_mask].copy()
model_test['model_probability'] = model.predict_proba(X_test)[:, 1]
model_test['model_prediction'] = model.predict(X_test)

# Rebuild the Week-4 baseline and align it to the same test rows by content_id.
base_df = df[['content_id', 'client_id', 'trend_direction', 'days_since_last_update', 'impressions_prev_30d', 'is_declining_label']].copy()
base_df['stale_points'] = np.select(
    [base_df['days_since_last_update'] >= 365, base_df['days_since_last_update'] >= 180],
    [2, 1],
    default=0,
)
base_df['visibility_points'] = np.select(
    [base_df['impressions_prev_30d'] >= 2000, base_df['impressions_prev_30d'] >= 100],
    [2, 1],
    default=0,
)
base_df['baseline_score'] = base_df['stale_points'] + base_df['visibility_points']
base_test = base_df[base_df['content_id'].isin(model_test['content_id'])].copy()
base_test = base_test.sort_values('baseline_score', ascending=False).reset_index(drop=True)
model_test = model_test.sort_values('model_probability', ascending=False).reset_index(drop=True)

# Compare the model and baseline as ranked top-K precision on the same holdout slice.
comparison_rows = []
for k in [10, 50, 100]:
    baseline_top = base_test.head(k)
    model_top = model_test.head(k)
    comparison_rows.append({
        'metric': f'precision_at_{k}',
        'baseline': round(float(baseline_top['is_declining_label'].mean()), 3),
        'logistic_regression': round(float(model_top['is_declining_label'].mean()), 3),
        'base_rate_test': round(float(y_test.mean()), 3),
    })

comparison = pd.DataFrame(comparison_rows)
print('Model ROC-AUC on held-out clients:', round(float(roc_auc_score(y_test, model_test['model_probability'])), 3))
print('\nComparison table: baseline vs logistic regression on the same holdout slice')
print(comparison.to_string(index=False))

# Inspect feature directions to make sure the model is reading sensible signals.
coefficients = pd.DataFrame({
    'feature': feature_columns,
    'coefficient': model.named_steps['logisticregression'].coef_[0],
})

coefficients['abs_coefficient'] = coefficients['coefficient'].abs()

print('\nTop model features by absolute coefficient:')
print(coefficients.sort_values('abs_coefficient', ascending=False).head(5)[['feature', 'coefficient']].to_string(index=False))

Model ROC-AUC on held-out clients: 0.493

Comparison table: baseline vs logistic regression on the same holdout slice
          metric  baseline  logistic_regression  base_rate_test
 precision_at_10      0.70                 0.50           0.618
 precision_at_50      0.56                 0.56           0.618
precision_at_100      0.54                 0.59           0.618

Top model features by absolute coefficient:
              feature  coefficient
 impressions_prev_30d     1.256430
      impressions_90d    -1.227116
           clicks_90d    -0.680496
    sessions_prev_30d     0.629878
days_with_impressions     0.568977


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Error analysis: where the model is wrong and what kinds of pages it over- or under-fires on.
errors = model_test[['content_id', 'client_id', 'is_declining_label', 'model_probability']].copy()
errors['error_type'] = np.where(
    (errors['model_probability'] >= 0.6) & (errors['is_declining_label'] == 0),
    'false_alarm',
    np.where(
        (errors['model_probability'] < 0.4) & (errors['is_declining_label'] == 1),
        'missed_decline',
        'reasonable',
    ),
)
print('Error counts by type:')
print(errors['error_type'].value_counts().to_string())

print('\nHighest-confidence false alarms (predicted decline but actually not declining):')
print(errors[errors['error_type'] == 'false_alarm'].sort_values('model_probability', ascending=False).head(5)[['content_id', 'model_probability', 'is_declining_label']].to_string(index=False))
print('\nMissed declines (actually declining but low model score):')
print(errors[errors['error_type'] == 'missed_decline'].sort_values('model_probability').head(5)[['content_id', 'model_probability', 'is_declining_label']].to_string(index=False))

print('\nInterpretation:')
print('The model is most likely to over-rank pages that have strong visibility but weak evidence of true decline, especially when the traffic pattern is noisy or short-lived. It is also more likely to miss declines when the page still has modest visibility and the content signal alone is not enough to separate it from non-declining pages.')
print('This is consistent with the lane: ranking position and visibility matter, but real decline remains a noisy operational outcome, so a simple model is a decision aid rather than a perfect predictor.')


Error counts by type:
error_type
reasonable        9645
false_alarm       2712
missed_decline     146

Highest-confidence false alarms (predicted decline but actually not declining):
          content_id  model_probability  is_declining_label
content_35e6edbb0a2d           1.000000                   0
content_abf535d9051e           1.000000                   0
content_44e481c8f55b           0.999999                   0
content_a965a1fc5544           0.999996                   0
content_d6b17274474e           0.999965                   0

Missed declines (actually declining but low model score):
          content_id  model_probability  is_declining_label
content_da8bd76fe1f8           0.024418                   1
content_7eb2b37dd0e5           0.043524                   1
content_9e0abcad7411           0.069559                   1
content_8180f9c9cbf0           0.091244                   1
content_79c22907284a           0.099932                   1

Interpretation:
The model is most lik

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.